# 08end_to_end_demo 노트북 목표
1. 저장된 프로젝트 제안 모델을 실제 리뷰 입력에 적용한다.
2. 리뷰 텍스트에서 KcBERT 임베딩과 메타데이터를 생성한다.
3. 이벤트 리뷰 확률, 이벤트 여부, 후보 2 별점 정제 가중치를 출력한다.
4. 예시 가게 리뷰 여러 개를 입력해 기존 평균 별점과 정제 후 별점을 비교한다.

이 노트북은 모델을 새로 학습하지 않는다. 01~07번에서 만든 `outputs/final_proposed_model.joblib`을 실제 입력에 적용하는 end-to-end 데모이다.

## 1. 라이브러리 로드

- 저장된 모델 bundle을 불러오고, KcBERT 임베딩과 입력 feature를 만들기 위한 패키지를 임포트한다.
- `rating`은 별점 정제 대상이므로 모델 입력 feature로 사용하지 않는다.

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import joblib

import emoji
from soynlp.normalizer import repeat_normalize

import torch
from transformers import AutoTokenizer, AutoModel

## 2. 저장 모델과 KcBERT 로드

- 06번에서 저장한 프로젝트 제안 모델 `final_proposed_model.joblib`만 사용한다.
- KcBERT는 02번과 동일하게 `beomi/kcbert-base`를 특징 추출기로 사용한다.
- 저장 bundle의 `feature_cols`를 그대로 사용해 학습 당시 입력 순서와 맞춘다.

In [2]:
MODEL_BUNDLE_PATH = Path('outputs/final_proposed_model.joblib')
KCBERT_MODEL_NAME = 'beomi/kcbert-base'

if not MODEL_BUNDLE_PATH.exists():
    raise FileNotFoundError(
        f'제안 모델 bundle이 없습니다: {MODEL_BUNDLE_PATH}. 01~07번을 먼저 실행하세요.'
    )

model_bundle = joblib.load(MODEL_BUNDLE_PATH)
event_model = model_bundle['model']
feature_cols = model_bundle.get('feature_cols')
threshold = float(model_bundle.get('best_threshold', model_bundle.get('default_threshold', 0.5)))

if feature_cols is None:
    raise ValueError('저장 모델 bundle에 feature_cols가 없습니다.')

expected_embedding_cols = [f'kcbert_{i}' for i in range(768)]
expected_meta_cols = ['text_length', 'emoji_count', 'photo_count']
expected_feature_cols = expected_embedding_cols + expected_meta_cols
missing_required_cols = [col for col in expected_feature_cols if col not in feature_cols]
if missing_required_cols:
    raise ValueError(f'제안 모델 feature_cols에 필요한 컬럼이 없습니다: {missing_required_cols[:10]}')

assert 'rating' not in feature_cols, 'rating은 별점 정제 대상이므로 모델 입력에 포함되면 안 됩니다.'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = AutoTokenizer.from_pretrained(KCBERT_MODEL_NAME)
kcbert_model = AutoModel.from_pretrained(KCBERT_MODEL_NAME).to(device)
kcbert_model.eval()

print('제안 모델:', model_bundle.get('model_name'))
print('feature set:', model_bundle.get('feature_set'))
print('feature 수:', len(feature_cols))
print('threshold:', round(threshold, 4))
print('KcBERT 실행 환경:', device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: beomi/kcbert-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


제안 모델: proposed_hybrid_mlp_04
feature set: kcbert_pca+metadata
feature 수: 771
threshold: 0.5009
KcBERT 실행 환경: cpu


## 3. 리뷰 전처리와 feature 생성 함수

- 01번과 같은 방식으로 리뷰 텍스트를 정제한다.
- 정제 텍스트는 KcBERT 입력으로 사용하고, 원문 리뷰에서 `text_length`, `emoji_count`, `photo_count`를 만든다.
- KcBERT `[CLS]` 벡터 768차원과 메타데이터 3개를 합쳐 모델 입력 feature를 만든다.

In [3]:
pattern = re.compile(r'[^ .,?!/@$%~％·∼()\x00-\x7F ㄱ-ㅣ가-힣]+')
url_pattern = re.compile(
    r'https?:\/\/(www\.)?[-a-zA-Z0-9@:%._\+~#=]{1,256}'
    r'\.[a-zA-Z0-9()]{1,6}\b'
    r'([-a-zA-Z0-9()@:%_\+.~#?&//=]*)'
)


def clean_review_text(text):
    text = str(text)
    text = url_pattern.sub('', text)
    text = emoji.replace_emoji(text, replace='')
    text = pattern.sub(' ', text)
    text = text.strip()
    text = repeat_normalize(text, num_repeats=2)
    return text


def count_emoji(text):
    return len([char for char in str(text) if char in emoji.EMOJI_DATA])


def normalize_photo_count(photo_count):
    if photo_count is None or pd.isna(photo_count):
        return 0
    return int(photo_count)


def embed_texts(cleaned_texts, batch_size=16):
    cls_embeddings = []
    with torch.no_grad():
        for start in range(0, len(cleaned_texts), batch_size):
            batch_text = cleaned_texts[start:start + batch_size]
            inputs = tokenizer(
                batch_text,
                return_tensors='pt',
                truncation=True,
                max_length=128,
                padding='max_length',
            ).to(device)
            outputs = kcbert_model(**inputs)
            cls_vector = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            cls_embeddings.append(cls_vector)

    return np.vstack(cls_embeddings)


def make_feature_frame(review_texts, photo_counts):
    if isinstance(review_texts, str):
        review_texts = [review_texts]
    if np.isscalar(photo_counts) or photo_counts is None:
        photo_counts = [photo_counts] * len(review_texts)

    if len(review_texts) != len(photo_counts):
        raise ValueError('review_texts와 photo_counts의 길이가 같아야 합니다.')

    rows = []
    cleaned_texts = []
    for review_text, photo_count in zip(review_texts, photo_counts):
        raw_text = str(review_text)
        cleaned_text = clean_review_text(raw_text)
        cleaned_texts.append(cleaned_text)
        rows.append({
            'review_text': raw_text,
            'cleaned_review_text': cleaned_text,
            'text_length': len(raw_text),
            'emoji_count': count_emoji(raw_text),
            'photo_count': normalize_photo_count(photo_count),
        })

    embeddings = embed_texts(cleaned_texts)
    embedding_df = pd.DataFrame(embeddings, columns=expected_embedding_cols)
    info_df = pd.DataFrame(rows)
    feature_df = pd.concat([embedding_df, info_df[expected_meta_cols]], axis=1)

    missing_model_cols = [col for col in feature_cols if col not in feature_df.columns]
    if missing_model_cols:
        raise KeyError(f'모델 입력 feature를 만들 수 없습니다: {missing_model_cols[:10]}')

    return feature_df[feature_cols], info_df

## 4. 단일 리뷰 이벤트 판별

- 리뷰 1개와 사진 개수를 입력하면 이벤트 리뷰 확률을 출력한다.
- 이벤트 확률이 threshold 이상이면 이벤트 리뷰로 판단한다.
- 후보 2 가중치는 `1 - 이벤트 확률`이며, 이벤트 가능성이 높을수록 별점 반영 비중이 낮아진다.

In [4]:
DEMO_REVIEW_TEXT = '리뷰 이벤트 참여합니다. 콜라 서비스 부탁드려요! 음식도 맛있었어요 😊'
DEMO_PHOTO_COUNT = 1

single_feature, single_info = make_feature_frame(DEMO_REVIEW_TEXT, DEMO_PHOTO_COUNT)
single_event_prob = float(event_model.predict_proba(single_feature)[:, 1][0])
single_event_pred = int(single_event_prob >= threshold)
single_candidate_2_weight = float(np.clip(1 - single_event_prob, 0, 1))

single_result = pd.DataFrame([{
    '원문 리뷰': single_info.loc[0, 'review_text'],
    '정제 리뷰': single_info.loc[0, 'cleaned_review_text'],
    '사진 수': single_info.loc[0, 'photo_count'],
    '텍스트 길이': single_info.loc[0, 'text_length'],
    '이모지 수': single_info.loc[0, 'emoji_count'],
    'threshold': threshold,
    '이벤트 확률': single_event_prob,
    '이벤트 예측': '이벤트 리뷰' if single_event_pred == 1 else '일반 리뷰',
    '후보 2 가중치': single_candidate_2_weight,
}]).round({
    'threshold': 4,
    '이벤트 확률': 4,
    '후보 2 가중치': 4,
})

assert 0 <= single_event_prob <= 1
assert np.isclose(single_candidate_2_weight, 1 - single_event_prob)

display(single_result)

,원문 리뷰,정제 리뷰,사진 수,텍스트 길이,이모지 수,threshold,이벤트 확률,이벤트 예측,후보 2 가중치
0,리뷰 이벤트 참여합니다. 콜라 서비스 부탁드려요! 음식도 맛있었어요 😊,리뷰 이벤트 참여합니다. 콜라 서비스 부탁드려요! 음식도 맛있었어요,1,39,1,0.5009,0.628,이벤트 리뷰,0.372


## 5. 가게 단위 별점 정제 데모

- 예시 가게의 여러 리뷰를 한 번에 입력한다.
- 각 리뷰의 이벤트 확률을 계산하고, 후보 2 가중 평균으로 가게 별점을 정제한다.
- 정제 공식은 `sum(rating * (1 - event_prob)) / sum(1 - event_prob)`이다.

In [5]:
demo_store_reviews = pd.DataFrame([
    {'review_text': '리뷰 이벤트 참여합니다 콜라 부탁드려요', 'rating': 5.0, 'photo_count': 1},
    {'review_text': '양 많고 맛있어요 다음에 또 시킬게요', 'rating': 5.0, 'photo_count': 0},
    {'review_text': '음식이 조금 식어서 왔어요 그래도 괜찮았습니다', 'rating': 3.0, 'photo_count': 0},
    {'review_text': '포토리뷰 약속했어요 서비스 감사합니다', 'rating': 5.0, 'photo_count': 2},
    {'review_text': '배달도 빠르고 포장도 깔끔했어요', 'rating': 4.0, 'photo_count': 0},
])

store_feature, store_info = make_feature_frame(
    demo_store_reviews['review_text'].tolist(),
    demo_store_reviews['photo_count'].tolist(),
)
store_event_prob = event_model.predict_proba(store_feature)[:, 1]
store_event_pred = (store_event_prob >= threshold).astype(int)
store_candidate_2_weight = np.clip(1 - store_event_prob, 0, 1)

demo_store_result = demo_store_reviews.copy()
demo_store_result['cleaned_review_text'] = store_info['cleaned_review_text']
demo_store_result['event_prob'] = store_event_prob
demo_store_result['event_pred'] = store_event_pred
demo_store_result['event_pred_name'] = np.where(store_event_pred == 1, '이벤트 리뷰', '일반 리뷰')
demo_store_result['candidate_2_weight'] = store_candidate_2_weight
demo_store_result['candidate_2_weighted_rating'] = demo_store_result['rating'] * demo_store_result['candidate_2_weight']

original_mean_rating = float(demo_store_result['rating'].mean())
weight_sum = float(demo_store_result['candidate_2_weight'].sum())
refined_rating = (
    original_mean_rating
    if weight_sum <= 0
    else float(demo_store_result['candidate_2_weighted_rating'].sum() / weight_sum)
)
rating_delta = refined_rating - original_mean_rating

store_summary = pd.DataFrame([{
    '리뷰 수': len(demo_store_result),
    '이벤트 예측 수': int(demo_store_result['event_pred'].sum()),
    '기존 평균 별점': original_mean_rating,
    '정제 후 별점': refined_rating,
    '별점 변화량': rating_delta,
}]).round({
    '기존 평균 별점': 4,
    '정제 후 별점': 4,
    '별점 변화량': 4,
})

detail_display = demo_store_result[[
    'review_text',
    'rating',
    'photo_count',
    'event_prob',
    'event_pred_name',
    'candidate_2_weight',
]].rename(columns={
    'review_text': '리뷰',
    'rating': '별점',
    'photo_count': '사진 수',
    'event_prob': '이벤트 확률',
    'event_pred_name': '이벤트 예측',
    'candidate_2_weight': '후보 2 가중치',
}).round({
    '이벤트 확률': 4,
    '후보 2 가중치': 4,
})

assert np.all((store_event_prob >= 0) & (store_event_prob <= 1))
np.testing.assert_allclose(store_candidate_2_weight, 1 - store_event_prob)

print('리뷰별 이벤트 확률과 후보 2 가중치')
display(detail_display)
print('가게 단위 후보 2 별점 정제 결과')
display(store_summary)

리뷰별 이벤트 확률과 후보 2 가중치


,리뷰,별점,사진 수,이벤트 확률,이벤트 예측,후보 2 가중치
0,리뷰 이벤트 참여합니다 콜라 부탁드려요,5.0,1,0.4172,일반 리뷰,0.5828
1,양 많고 맛있어요 다음에 또 시킬게요,5.0,0,0.4131,일반 리뷰,0.5869
2,음식이 조금 식어서 왔어요 그래도 괜찮았습니다,3.0,0,0.3113,일반 리뷰,0.6887
3,포토리뷰 약속했어요 서비스 감사합니다,5.0,2,0.2958,일반 리뷰,0.7042
4,배달도 빠르고 포장도 깔끔했어요,4.0,0,0.1834,일반 리뷰,0.8166


가게 단위 후보 2 별점 정제 결과


,리뷰 수,이벤트 예측 수,기존 평균 별점,정제 후 별점,별점 변화량
0,5,0,4.4,4.3507,-0.0493


## 6. 결과 해석

- 이벤트 확률이 높은 리뷰는 후보 2 가중치가 낮아져 별점 평균에 덜 반영된다.
- 이벤트 확률이 낮은 리뷰는 일반 리뷰에 가깝다고 보고 별점 영향력을 더 유지한다.
- 이 노트북은 최종 모델 적용 데모이며, 모델 학습과 성능 비교는 04~07번에서 수행한다.